In [1]:
!pip install -q datasets
!pip install -q -U transformers accelerate
import pandas as pd
import torch
import unicodedata
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset


In [2]:
# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True
# torch.set_float32_matmul_precision("high")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("GPU:", torch.cuda.get_device_name(0))

In [3]:
import pandas as pd

file_path = "/kaggle/input/datasets/tanujsaxena/sandhi-data/sandhi_data.xlsx"

df = pd.read_excel(file_path)

print(df.head())
print(df.columns)
print("Total samples:", len(df))

                                Word                               Split
0                        प्रथमोऽङ्कः                        प्रथमः+अङ्कः
1                            शब्द इव                            शब्दः+इव
2                             इत इतः                             इतः+इतः
3                            कुतो नु                             कुतः+नु
4  खल्वेष समुत्थितो समुत्थितो ध्वनिः  खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः
Index(['Word', 'Split'], dtype='object')
Total samples: 13930


In [4]:
def clean_text(text):
    text = str(text).strip()
    text = unicodedata.normalize("NFC", text)
    text = " ".join(text.split())
    return text

def preprocess_pair(inp, out):
    inp = clean_text(inp)
    out = clean_text(out).replace("+", " ")
    inp = "split: " + inp
    return inp, out

data = []
for _, row in df.iterrows():
    inp, out = preprocess_pair(row[0], row[1])
    data.append((inp, out))

print("Before filtering:", len(data))

filtered_data = []
for inp, out in data:
    if " " not in inp.replace("split: ", "").strip():
        filtered_data.append((inp, out))

data = filtered_data
print("After filtering:", len(data))


/tmp/ipykernel_24/4216803760.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  inp, out = preprocess_pair(row[0], row[1])


Before filtering: 13930
After filtering: 9018


In [5]:
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))


Train: 7214
Val: 902
Test: 902


In [6]:
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))


Train: 7214
Val: 902
Test: 902


In [7]:
train_dataset = HFDataset.from_list(
    [{"input_text": inp, "target_text": out} for inp, out in train_data]
)

val_dataset = HFDataset.from_list(
    [{"input_text": inp, "target_text": out} for inp, out in val_data]
)


In [8]:
model_name = "google/byt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("GPU:", torch.cuda.get_device_name(0))


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

GPU: Tesla P100-PCIE-16GB


In [9]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=80,
        truncation=True
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=80,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [10]:
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)


Map:   0%|          | 0/7214 [00:00<?, ? examples/s]

Map:   0%|          | 0/902 [00:00<?, ? examples/s]

In [11]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)


In [12]:
training_args = TrainingArguments(
    output_dir="./byt5_sandhi",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate = 5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    dataloader_num_workers=4,
    load_best_model_at_end=True,
    report_to="none",
    max_grad_norm=1.0,
    save_only_model=True,
    save_total_limit=1
)


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.182886,0.120089
2,0.118991,0.079464
3,0.094484,0.067954
4,0.083633,0.062790
5,0.080657,0.058879
6,0.070407,0.057908
7,0.074361,0.055203
8,0.064895,0.054100
9,0.064578,0.053968
10,0.056091,0.053532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2260, training_loss=0.14695035282489471, metrics={'train_runtime': 1726.8687, 'train_samples_per_second': 41.775, 'train_steps_per_second': 1.309, 'total_flos': 9873862717054464.0, 'train_loss': 0.14695035282489471, 'epoch': 10.0})

In [14]:
trainer.save_model("./best_byt5_model")
tokenizer.save_pretrained("./best_byt5_model")

torch.save(trainer.model.state_dict(), "byt5_best_model.pt")

print("Model saved successfully.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.


In [15]:
def predict(text):
    model.eval()
    text = "split: " + clean_text(text)

    inputs = tokenizer(text, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=80,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [16]:
def evaluate_exact(dataset):
    correct = 0
    total = len(dataset)

    for example in dataset:
        input_ids = torch.tensor(example["input_ids"]).unsqueeze(0).to(device)

        outputs = model.generate(
            input_ids=input_ids,
            max_length=64
        )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

        gold_ids = [id for id in example["labels"] if id != -100]
        gold = tokenizer.decode(gold_ids, skip_special_tokens=True)

        if pred.strip() == gold.strip():
            correct += 1

    return correct / total

val_exact = evaluate_exact(val_dataset)
print("Validation Exact Match:", val_exact)


Validation Exact Match: 0.7017738359201774


In [17]:
def show_errors(dataset, n=7):
    errors = 0
    for example in dataset:
        input_ids = torch.tensor(example["input_ids"]).unsqueeze(0).to(device)
        # outputs = model.generate(input_ids=input_ids, max_length=64)
        outputs = model.generate(
    input_ids=input_ids,
    max_length=64,
    num_beams=4,
    early_stopping=True
)


        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        gold_ids = [id for id in example["labels"] if id != -100]
        gold = tokenizer.decode(gold_ids, skip_special_tokens=True)

        if pred.strip() != gold.strip():
            print("INPUT :", tokenizer.decode(example["input_ids"], skip_special_tokens=True))
            print("PRED  :", pred)
            print("GOLD  :", gold)
            print("------")
            errors += 1
            if errors >= n:
                break

show_errors(val_dataset)


INPUT : split: निराशीरपरिग्रहः
PRED  : निराशीः अपरिग्रहः
GOLD  : निः आशीः अपरिग्रहः
------
INPUT : split: लक्षणादिति
PRED  : लक्षण इति
GOLD  : लक्षणात् इति
------
INPUT : split: शान्तिस्तथा
PRED  : शान्तिः तथा
GOLD  : शान्तः तथा
------
INPUT : split: संयमोऽपि
PRED  : संयमः अपि
GOLD  : संयम अपि
------
INPUT : split: उभयोरपि
PRED  : उभयः अपि
GOLD  : उभयोः अपि
------
INPUT : split: नाभ्यस्ताच्छतुः
PRED  : न अभ्यः तात् छतुः
GOLD  : न अभ्यस्तात् शतुः
------
INPUT : split: श्रृगालीव
PRED  : श्रृगाली इव
GOLD  : शृगाली इव
------


In [18]:
def char_accuracy(dataset):
    correct = 0
    total = 0

    for example in dataset:
        input_ids = torch.tensor(example["input_ids"]).unsqueeze(0).to(device)
        # outputs = model.generate(input_ids=input_ids, max_length=64)
        outputs = model.generate(
    input_ids=input_ids,
    max_length=64,
    num_beams=4,
    early_stopping=True
)


        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        gold_ids = [id for id in example["labels"] if id != -100]
        gold = tokenizer.decode(gold_ids, skip_special_tokens=True)

        min_len = min(len(pred), len(gold))
        for i in range(min_len):
            if pred[i] == gold[i]:
                correct += 1
        total += max(len(pred), len(gold))

    return correct / total

print("Character Accuracy:", char_accuracy(val_dataset))


Character Accuracy: 0.807815210444627
